# Demo guiada — UX para consumo de modelos

La semana 5 tenía formulario y predicción. Ahora añadimos estados, errores, confianza y telemetría mínima sin importar la app de solución.


## 1. Estado explícito

Una interfaz no debería mostrar sólo una predicción. También debe saber si está esperando, si falló y qué confianza tiene el resultado.


In [ ]:
from dataclasses import dataclass, field
from time import perf_counter

@dataclass
class PredictionState:
    status: str = "idle"  # idle, loading, success, error
    message: str = "Esperando entrada"
    prediction: str | None = None
    confidence: float | None = None
    latency_ms: float | None = None

@dataclass
class Telemetry:
    total_requests: int = 0
    successful_requests: int = 0
    failed_requests: int = 0
    latencies_ms: list[float] = field(default_factory=list)

state = PredictionState()
telemetry = Telemetry()
print(state)

## 2. Gateway y controlador sin Streamlit

**TODO 1:** decide qué umbral convierte una confianza en mensaje de cautela. Importa porque UX debe comunicar incertidumbre. Inspecciona funciones pequeñas y `dataclass`. Verifica que el estado termina en `success`.


In [ ]:
class DemoGateway:
    def __init__(self, confidence: float = 0.74):
        self.confidence = confidence

    def predict(self, values: dict) -> dict:
        if values["alcohol"] < 0:
            raise ValueError("alcohol fuera de rango")
        return {"label": "class_1", "confidence": self.confidence}


def submit_prediction(values: dict, gateway: DemoGateway, telemetry: Telemetry) -> PredictionState:
    telemetry.total_requests += 1
    started = perf_counter()
    try:
        result = gateway.predict(values)
        latency_ms = (perf_counter() - started) * 1000
        telemetry.successful_requests += 1
        telemetry.latencies_ms.append(latency_ms)
        confidence = result["confidence"]
        message = "Predicción lista" if confidence >= 0.70 else "Predicción con confianza baja: revisa antes de actuar"
        return PredictionState("success", message, result["label"], confidence, latency_ms)
    except Exception as exc:
        latency_ms = (perf_counter() - started) * 1000
        telemetry.failed_requests += 1
        telemetry.latencies_ms.append(latency_ms)
        return PredictionState("error", f"No se pudo predecir: {exc}", latency_ms=latency_ms)

SAMPLE = {"alcohol": 13.2, "color_intensity": 5.6}
state = submit_prediction(SAMPLE, DemoGateway(confidence=0.74), telemetry)
print(state)
assert state.status == "success"
assert telemetry.successful_requests == 1

## 3. Error y recuperación

**TODO 2:** añade otro error de entrada. Decide qué mensaje sería útil para un usuario no técnico. Inspecciona el bloque `try/except`. Verifica que un error no borra la telemetría acumulada.


In [ ]:
previous_success = state
error_state = submit_prediction({"alcohol": -1, "color_intensity": 5.6}, DemoGateway(), telemetry)
print(error_state)
assert error_state.status == "error"
assert telemetry.failed_requests == 1
assert previous_success.status == "success"

## 4. Limpiar pantalla sin borrar observabilidad

Limpiar la última tarjeta visual no debe borrar contadores, porque la observabilidad resume el uso de la sesión.


In [ ]:
def clear_last_state() -> PredictionState:
    return PredictionState()

state = clear_last_state()
print({"state": state, "telemetry": telemetry})
assert state.status == "idle"
assert telemetry.total_requests == 2
assert len(telemetry.latencies_ms) == 2

## 5. Preguntas para el debrief

1. ¿Qué diferencia hay entre un recurso cacheado y una respuesta cacheada?
2. ¿Por qué una confianza baja no debe mostrarse como un error técnico?
3. ¿Qué métrica de telemetría añadirías antes de producción?
4. ¿Qué parte probarías sin abrir Streamlit?
